# Lesson 12D: Generative Adversarial Networks - Practical

## Training GANs on 2D Data

**Case Study**: Train a GAN to learn the distribution of a complex 2D dataset (moons). Watch as the generator learns to fool the discriminator!

**The GAN game**: Generator creates fake data, Discriminator tries to detect fakes.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons

torch.manual_seed(42)
np.random.seed(42)
print('✅ Loaded!')

## Step 1: Define Generator and Discriminator

**Generator**: Maps noise z → fake data x  
**Discriminator**: Classifies real (1) vs fake (0)

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=10, hidden_dim=64, output_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, z):
        return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.net(x)

G = Generator(latent_dim=10, hidden_dim=64, output_dim=2)
D = Discriminator(input_dim=2, hidden_dim=64)

print('✅ GAN initialized')
print(f'   Generator params: {sum(p.numel() for p in G.parameters())}')
print(f'   Discriminator params: {sum(p.numel() for p in D.parameters())}')

## Step 2: Prepare Real Data

We'll use the moons dataset as our target distribution.

In [ ]:
# Generate real data
X_real, _ = make_moons(n_samples=2000, noise=0.05, random_state=42)

print(f"Real data shape: {X_real.shape}")
print(f"Data range: [{X_real.min():.2f}, {X_real.max():.2f}]")

# Visualize real data
plt.figure(figsize=(8, 6))
plt.scatter(X_real[:, 0], X_real[:, 1], s=20, alpha=0.6, c='blue', label='Real data')
plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.title('Real Data Distribution (Moons)', fontweight='bold', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Convert to tensor
X_real = torch.FloatTensor(X_real)
print('✅ Data prepared!')

## Step 3: Define Loss and Optimizers

**Discriminator loss**: Maximize log(D(x_real)) + log(1 - D(G(z)))  
**Generator loss**: Maximize log(D(G(z))) ≈ minimize log(1 - D(G(z)))

In [ ]:
criterion = nn.BCELoss()
opt_G = optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))

print('✅ Optimizers configured')
print('   Using Adam with lr=0.0002')
print('   BCE loss for adversarial training')

## Step 4: Train GAN

Training alternates between D and G updates.

In [ ]:
# Training setup
batch_size = 64
n_epochs = 1000
latent_dim = 10

# Storage for losses
d_losses = []
g_losses = []

print("Training GAN...")
for epoch in range(n_epochs):
    # === Train Discriminator ===
    # Real samples
    idx = torch.randint(0, len(X_real), (batch_size,))
    real_samples = X_real[idx]
    real_labels = torch.ones(batch_size, 1)
    
    # Fake samples
    z = torch.randn(batch_size, latent_dim)
    fake_samples = G(z)
    fake_labels = torch.zeros(batch_size, 1)
    
    # Discriminator loss
    D_real = D(real_samples)
    D_fake = D(fake_samples.detach())
    loss_D_real = criterion(D_real, real_labels)
    loss_D_fake = criterion(D_fake, fake_labels)
    loss_D = loss_D_real + loss_D_fake
    
    opt_D.zero_grad()
    loss_D.backward()
    opt_D.step()
    
    # === Train Generator ===
    z = torch.randn(batch_size, latent_dim)
    fake_samples = G(z)
    D_fake = D(fake_samples)
    loss_G = criterion(D_fake, real_labels)  # Want D to output 1 for fakes
    
    opt_G.zero_grad()
    loss_G.backward()
    opt_G.step()
    
    # Log
    d_losses.append(loss_D.item())
    g_losses.append(loss_G.item())
    
    if (epoch + 1) % 200 == 0:
        print(f'Epoch {epoch+1}/{n_epochs}')
        print(f'  D Loss: {loss_D.item():.4f}, G Loss: {loss_G.item():.4f}')
        print(f'  D(real): {D_real.mean().item():.4f}, D(fake): {D_fake.mean().item():.4f}')

print('✅ GAN trained!')

## Step 5: Visualize Training Progress

In [ ]:
# Plot losses
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(d_losses, label='Discriminator', linewidth=2, alpha=0.7)
ax.plot(g_losses, label='Generator', linewidth=2, alpha=0.7)
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('GAN Training Losses', fontweight='bold', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

# Plot discriminator confidence over time
window = 100
D_real_avg = [np.mean(d_losses[max(0, i-window):i+1]) for i in range(len(d_losses))]

ax = axes[1]
ax.plot(D_real_avg, linewidth=2, color='purple')
ax.axhline(y=0.693, color='red', linestyle='--', linewidth=2, 
          label='Optimal (log(2) ≈ 0.693)')
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('D Loss (moving avg)', fontsize=12)
ax.set_title('Discriminator Convergence', fontweight='bold', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('✅ Training progress visualized!')

## Step 6: Compare Real vs Generated

In [ ]:
# Generate samples
G.eval()
with torch.no_grad():
    z = torch.randn(2000, latent_dim)
    fake_samples = G(z).numpy()

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Real data
ax = axes[0]
ax.scatter(X_real[:, 0], X_real[:, 1], s=20, alpha=0.6, c='blue')
ax.set_xlabel('Feature 1', fontsize=11)
ax.set_ylabel('Feature 2', fontsize=11)
ax.set_title('Real Data', fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3)

# Generated data
ax = axes[1]
ax.scatter(fake_samples[:, 0], fake_samples[:, 1], s=20, alpha=0.6, c='red')
ax.set_xlabel('Feature 1', fontsize=11)
ax.set_ylabel('Feature 2', fontsize=11)
ax.set_title('Generated Data', fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3)

# Overlay
ax = axes[2]
ax.scatter(X_real[:1000, 0], X_real[:1000, 1], s=20, alpha=0.4, c='blue', label='Real')
ax.scatter(fake_samples[:1000, 0], fake_samples[:1000, 1], s=20, alpha=0.4, c='red', label='Generated')
ax.set_xlabel('Feature 1', fontsize=11)
ax.set_ylabel('Feature 2', fontsize=11)
ax.set_title('Real vs Generated (Overlay)', fontweight='bold', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('✅ GAN successfully learned the distribution!')

## Summary

**GAN Training Insights**:
- ✅ Generator learned to mimic the moon distribution
- ✅ Discriminator couldn't distinguish real from fake (equilibrium)
- ✅ No explicit loss on real data distribution needed

**Training Tips**:
1. **Balance**: D and G should improve together (neither dominates)
2. **Learning rates**: Use small LR (0.0001-0.0002) with Adam
3. **Architecture**: LeakyReLU for D, ReLU for G
4. **Stability**: Use label smoothing, gradient penalty for better training

**Production Applications**:
- Image generation (StyleGAN, BigGAN)
- Data augmentation
- Super-resolution
- Domain transfer (CycleGAN)

**Common Issues**:
- Mode collapse: G produces limited variety
- Vanishing gradients: G loss becomes unhelpful
- Training instability: Use Wasserstein GAN (WGAN) for stability